# Linear Projection (GEMM)
*Matrix multiplication — the most compute-intensive operation in a transformer.*

**Companion kernels:** `v0_naive.py` → `sm89_v1_smem_tiled.py` → `sm89_v2_tensor_core_mma.py`


## What Is a Linear Projection?

**In plain English:** A linear layer multiplies an input matrix by a weight matrix. This is how the model transforms representations: attention projections (Q, K, V, output), feedforward up/down projections — every "thinking" step is a matrix multiply.

**Where it appears:** Every attention sublayer (4× matmuls for Q, K, V, output projection) + every FFN sublayer (2-3× matmuls). ~70% of a transformer's total FLOPs are matrix multiplies.


In [ ]:
import math, random
random.seed(42)
print('ready')

## Worked Example: $2 \times 3$ Output from $2 \times 2$ Input

$$X = \begin{bmatrix} 1.0 & 2.0 \\ 3.0 & 4.0 \end{bmatrix}, \quad W = \begin{bmatrix} 0.5 & 0.5 \\ 0.3 & -0.2 \\ -0.1 & 0.8 \end{bmatrix}$$

We compute $Y = X W^T$ (PyTorch `F.linear` convention: W rows = output directions).

$$Y_{m,n} = \sum_{k=0}^{K-1} X_{m,k} \cdot W_{n,k}$$

| $Y[m,n]$ | Dot product | Result |
|-----------|-------------|--------|
| $Y[0,0]$ | $1.0×0.5 + 2.0×0.5$ | **1.5** |
| $Y[0,1]$ | $1.0×0.3 + 2.0×(-0.2)$ | **-0.1** |
| $Y[0,2]$ | $1.0×(-0.1) + 2.0×0.8$ | **1.5** |
| $Y[1,0]$ | $3.0×0.5 + 4.0×0.5$ | **3.5** |
| $Y[1,1]$ | $3.0×0.3 + 4.0×(-0.2)$ | **0.1** |
| $Y[1,2]$ | $3.0×(-0.1) + 4.0×0.8$ | **2.9** |


In [ ]:
X = [[1.0, 2.0], [3.0, 4.0]]
W = [[0.5, 0.5], [0.3, -0.2], [-0.1, 0.8]]
M, K, N = 2, 2, 3

Y = [[sum(X[m][k] * W[n][k] for k in range(K)) for n in range(N)] for m in range(M)]

print("Y = X @ W.T:")
for m, row in enumerate(Y):
    print(f"  row {m}: {[round(v,4) for v in row]}")

# Spot-check
assert abs(Y[0][0] - 1.5) < 1e-9 and abs(Y[1][2] - 2.9) < 1e-9
print("\n✓ Spot-checks pass")


## The Formula

$$\boxed{Y_{m,n} = \sum_{k=0}^{K-1} X_{m,k} \cdot W_{n,k}}$$

Equivalently: $Y = X W^T$ where $X \in \mathbb{R}^{M \times K}$, $W \in \mathbb{R}^{N \times K}$, $Y \in \mathbb{R}^{M \times N}$.

| Symbol | Meaning | In our example |
|--------|---------|----------------|
| $M$ | Batch size (number of input vectors) | 2 |
| $N$ | Output dimension (number of neurons) | 3 |
| $K$ | Input dimension | 2 |
| $X_{m,k}$ | Element $k$ of input vector $m$ | $X[0,0]=1.0$ |
| $W_{n,k}$ | Weight for output $n$ from input $k$ | $W[0,0]=0.5$ |

### 🗣️ Say It Out Loud
> *"Y at row m, column n equals the sum over k of X at m-comma-k times W at n-comma-k. This is the dot product of input row m with weight row n."*

**Tutor's gloss:** "Think of each row of W as a 'detector'. $W[0,:]$ detects the pattern $[0.5, 0.5]$ — it responds to the average of the two input features. The dot product with $X[0,:]=[1,2]$ gives $0.5×1 + 0.5×2 = 1.5$ — the detector fired."


## Arithmetic Intensity and the Cache Problem

**Arithmetic intensity** = FLOPs / bytes transferred.

For one $M \times N \times K$ matrix multiply:
$$I = \frac{2MNK}{4(MK + NK + MN)} \approx \frac{MNK}{2(MK + NK + MN)}$$

For large square matrices ($M = N = K = n$):
$$I \approx \frac{n^3}{6n^2} = \frac{n}{6} \text{ FLOP/byte}$$

| Matrix size $n$ | Intensity | Status on RTX 4080 (ridge ≈ 230) |
|----------------|-----------|----------------------------------|
| 64 | 10.7 FLOP/byte | Memory-bound (100× below ridge) |
| 1024 | 170 FLOP/byte | Approaching ridge |
| 4096 | 682 FLOP/byte | Strongly compute-bound |

**The cache problem:** In `v0_naive`, each thread fetches a different row of $W$ from HBM — zero reuse. Row $X[m,:]$ is re-fetched once per output column $n$. With $N=4096$ output columns, that's 4096 redundant HBM reads of the same row.

**The tiling fix:** All $B_M \times B_N$ threads in a CTA load the *same* $B_M \times B_K$ tile of X and $B_N \times B_K$ tile of W into shared memory once, then compute from SMEM (30–100× faster than HBM).


## Optimization Ladder

| Version | Compute path | Reuse | Approx. TFLOPS |
|---------|-------------|-------|----------------|
| `v0_naive` | CUDA scalar FMA | None | < 0.1 |
| `sm89_v1_smem_tiled` | CUDA scalar FMA | $B_M$ and $B_N$ | 1–5 |
| `sm89_v2_tensor_core_mma` | m16n8k16 TC instruction | $B_M \times B_N$ | 50–165 |

**Tensor Core instruction (SM89):**
`D[16,8] += A[16,16] × B[16,8]` in fp16 — 4096 FLOPs per warp per cycle.
In CuTeDSL: `cute.gemm(tiled_mma, rA, rB, rC)` emits this PTX.


## RTX 4080: Two Very Different GEMM Regimes

The same matrix multiply formula hits the RTX 4080 in two completely different ways depending on batch size.

**Decode mode (M=1, one token at a time):**
One input row × weight matrix = a GEMV (vector-matrix multiply).
- Data moved: read W (N×K values) + read x (K values) + write y (N values)
- FLOPs: 2·N·K
- For q_proj (N=K=4096): 2·4096·4096 FLOPs / (2·4096·4096·2 bytes) ≈ **0.5 FLOP/byte**
- Far below the ridge (151 FLOP/byte) → **memory-bound**, tensor cores sit idle
- Speedup path: FP8 cuts weight bytes in half → 2× faster reads

**Prefill mode (M=seq_len, processing the prompt):**
Full matrix multiply. As M grows, reuse of W improves.
- At M=512: intensity ≈ 512·4096·4096 / (512·4096·2 + 4096·4096·2 + 512·4096·2) bytes ≈ **85 FLOP/byte** (approaching ridge)
- At M=2048: intensity ≈ **340 FLOP/byte** → firmly compute-bound
- Speedup path: maximize tensor core utilization (tiling, async copies, persistent kernels)

| Regime | M | Intensity | Bound | Key speedup |
|--------|---|-----------|-------|-------------|
| Decode | 1 | 0.5 F/B | Memory | FP8 weights (2×) |
| Prefill small | 32 | 10 F/B | Memory | Better tiling |
| Prefill large | 2048 | 340 F/B | Compute | Tensor cores (57.5 TFLOPS) |

## CuTeDSL: Tensor Core MMA Walkthrough

The jump from `v1_smem_tiled` to `v2_tensor_core_mma` is the biggest speedup in the whole ladder.
Here is what that looks like:

```python
# Define the MMA "atom" — the smallest tensor core operation available on SM89
# SM89_16x8x16: processes a 16×16 fragment of A and 8×16 fragment of B
# Result: a 16×8 fragment of C, accumulated in FP32
tiled_mma = cute.make_tiled_mma(
    cute.SM89_16x8x16_F32BF16BF16F32_TN()
    # Read this name as: 16 output rows, 8 output cols, 16 inner depth
    # F32   = accumulator type (FP32 for precision)
    # BF16  = A matrix type
    # BF16  = B matrix type
    # F32   = output type
    # T = A is transposed in memory, N = B is not
)

@cute.kernel
def gemm_tc_kernel(mX, mW, mOut):
    # Each CTA (block) owns a [Bm × Bn] tile of the output
    rC = cute.partition_fragment_C(tiled_mma, ...)  # output accumulator in registers

    for k_tile in range(K // Bk):
        # Step 1: cooperatively load a [Bm × Bk] tile of X into shared memory
        # All 128+ threads in the CTA participate — each loads a few elements
        cute.copy(mX[row_tile, k_tile], sX)   # HBM → SMEM
        cute.copy(mW[col_tile, k_tile], sW)   # HBM → SMEM
        cute.sync_threads()  # everyone must finish loading before computing

        # Step 2: each thread loads its register fragment from shared memory
        cute.copy(sX, rA)
        cute.copy(sW, rB)

        # Step 3: fire the tensor core instruction
        # This single line compiles to PTX `mma.sync.aligned.m16n8k16`
        # One GPU clock cycle, 32 threads acting in lockstep → 4,096 FLOPs
        cute.gemm(tiled_mma, rA, rB, rC)   # rC += rA × rB

        cute.sync_threads()

    # Write the result from registers back to HBM
    cute.copy(rC, mOut[row_tile, col_tile])
```

**Why this is so much faster than scalar FMA:**

In `v1_smem_tiled`, each thread does its own multiply-adds one at a time.
In `v2_tensor_core_mma`, a group of 32 threads (a "warp") all fire ONE instruction together that does a 16×8×16 matrix multiply in hardware.

The SM89 tensor core is custom silicon designed to do exactly this — multiply two small matrices in one shot. No branching, no loop overhead, just pure math.

The RTX 4080 has 76 SMs × ~4 tensor core units each = hundreds of these firing simultaneously.
At peak: 57.5 TFLOPS BF16. With FP8: 115 TOPS (the operands are half the size so twice as many fit per cycle).

## Tile Sizing for Qwen3-8B on RTX 4080

Choosing tile dimensions is the most important decision in a GEMM kernel. Too small and you don't get enough data reuse. Too large and you can't fit in shared memory or run enough CTAs in parallel.

**RTX 4080 constraints:**
- 128 KB shared memory per SM (configurable — can dedicate up to ~100 KB to SMEM)
- 255 registers per thread, up to 1024 threads per SM
- 76 SMs total

**Standard tile for BF16 GEMM on SM89:** BM=128, BN=128, BK=32

```
SMEM per CTA (single buffer):
  A tile [BM=128, BK=32] in BF16  = 128×32×2 = 8 KB
  B tile [BN=128, BK=32] in BF16  = 128×32×2 = 8 KB
  Total: 16 KB

With double buffering (overlap load with compute):
  A×2 + B×2 = 32 KB
  128 KB / 32 KB = 4 CTAs per SM  ← good occupancy
```

**Worked example: q_proj at prefill (M=2048, N=4096, K=4096)**
```
Grid: (M/BM, N/BN) = (2048/128, 4096/128) = (16, 32) = 512 CTAs
K-tiles per CTA: K/BK = 4096/32 = 128 tiles
Each CTA: 128×128 output elements, 128 K-tiles
Total MMA ops per CTA: (128/16) × (128/8) × (128/16) = 8×16×8 = 1024 m16n8k16 ops
76 SMs × 4 CTAs = 304 active at once  →  512/304 ≈ 2 waves → GPU nearly fully utilized
```

**Worked example: k_proj at decode (M=1, N=1024, K=4096)**

This is a GEMV. The standard tiled GEMM doesn't help here — a 128×128 output tile with M=1 leaves 127/128 of the tile empty.

**Suggestion: split-K for decode GEMV**
```
Split K=4096 into 4 chunks of 1024. Each chunk is one "split":
  Split 0: computes partial y[0:1024] using W[0:1024, 0:4096/4]
  Split 1: computes partial y[0:1024] using W[0:1024, 1024:2048]
  ...
4 CTAs run in parallel, each computing a partial dot product over K/4 elements.
Then one final reduction: add the 4 partial results.

Result: 4× more parallelism → 4× better SM utilization during decode.
```

| Shape | M | N | K | Strategy | CTAs | Notes |
|-------|---|---|---|----------|------|-------|
| q_proj prefill | 2048 | 4096 | 4096 | Tiled (128×128) | 512 | GPU saturated |
| q_proj decode | 1 | 4096 | 4096 | Split-K (4 splits) | 4×32 = 128 | Acceptable |
| k_proj decode | 1 | 1024 | 4096 | Split-K (4 splits) | 4×8 = 32 | Underutilized |
| down_proj decode | 1 | 4096 | 12288 | Split-K (12 splits) | 12×32 = 384 | Good |

## Double Buffering: Overlapping Memory and Compute

The RTX 4080 supports `cp.async` — an instruction that copies data from global memory to
shared memory in the background, while the SM continues computing.

The idea: tensor cores compute with tile N while the memory system fetches tile N+1.
When the compute finishes, tile N+1 is already in SMEM. No waiting.

```
Without double buffering:        With double buffering:
  Load tile 0                      Load tile 0
  Compute tile 0                   Compute tile 0  ←─ parallel ─→  Load tile 1
  Load tile 1                      Compute tile 1  ←─ parallel ─→  Load tile 2
  Compute tile 1                   ...
  ...
```

**CuTeDSL pipeline code:**
```python
# Two-stage pipeline: ping-pong between sA_0/sB_0 and sA_1/sB_1
pipeline = cute.make_pipeline(stages=2, smem_a=sA_double, smem_b=sB_double)

# Prologue: kick off the first tile load before entering the loop
cute.pipeline_producer_acquire(pipeline, 0)
cute.copy(gmem_tile_A[0], pipeline.smem_a(0))
cute.copy(gmem_tile_B[0], pipeline.smem_b(0))
cute.pipeline_producer_commit(pipeline, 0)

for k_tile in range(num_k_tiles):
    cute.pipeline_consumer_wait(pipeline, k_tile)   # wait for this tile to land

    # Load tile from SMEM into registers using ldmatrix
    cute.copy(smem_to_rmem_copy, pipeline.smem_a(k_tile), rA)
    cute.copy(smem_to_rmem_copy, pipeline.smem_b(k_tile), rB)

    # Start fetching the next tile in the background
    if k_tile + 1 < num_k_tiles:
        cute.pipeline_producer_acquire(pipeline, k_tile + 1)
        cute.copy(gmem_tile_A[k_tile+1], pipeline.smem_a(k_tile+1))
        cute.copy(gmem_tile_B[k_tile+1], pipeline.smem_b(k_tile+1))
        cute.pipeline_producer_commit(pipeline, k_tile + 1)

    cute.gemm(tiled_mma, rA, rB, rC)           # tensor core fires here

    cute.pipeline_consumer_release(pipeline, k_tile)

cute.copy(rC, mOut[row_tile, col_tile])
```

**ldmatrix — the instruction that feeds tensor cores:**
Before `cute.gemm` fires, each thread needs its A and B fragments in registers,
arranged in the exact byte layout the tensor core hardware expects.
The instruction `ldmatrix.sync.aligned.m8n8.x4.shared.b16` loads four 8×8 BF16 matrices
from SMEM into registers for the entire warp in one shot.
In CuTeDSL, `smem_to_rmem_copy` above generates ldmatrix automatically — you don't write PTX by hand.

**RTX 4080 numbers with BM=128, BN=128, BK=32, double buffered:**
- SMEM per CTA: 2 × 16 KB = 32 KB
- Registers per thread: ~128 (rA=16, rB=8, rC=32, plus overhead)
- CTAs per SM: 128 KB ÷ 32 KB = 4 — the target occupancy for SM89 BF16 GEMMs